In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("umerkk12/online-retail-business")

print("Path to dataset files:", path)

100%|██████████| 7.22M/7.22M [00:01<00:00, 3.84MB/s]

Extracting model files...


Path to dataset files: /home/pccoe/.cache/kagglehub/datasets/umerkk12/online-retail-business/versions/1


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv("OnlineRetail.csv", encoding="latin1")

# Display first five records
print(df.head())

# Display dataset dimensions
print("Dataset Shape:", df.shape)



  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

        InvoiceDate  UnitPrice  CustomerID         Country  
0  01-12-2010 08:26       2.55     17850.0  United Kingdom  
1  01-12-2010 08:26       3.39     17850.0  United Kingdom  
2  01-12-2010 08:26       2.75     17850.0  United Kingdom  
3  01-12-2010 08:26       3.39     17850.0  United Kingdom  
4  01-12-2010 08:26       3.39     17850.0  United Kingdom  
Dataset Shape: (541909, 8)


## Find missing values

In [15]:
# Count missing values in each column
print(df.isnull().sum())

# Calculate missing value percentage
missing_percentage = (df.isnull().sum() / len(df)) * 100

print(missing_percentage)

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64
InvoiceNo       0.000000
StockCode       0.000000
Description     0.268311
Quantity        0.000000
InvoiceDate     0.000000
UnitPrice       0.000000
CustomerID     24.926694
Country         0.000000
dtype: float64


### Handling missing values:

In [17]:
# Fill missing categorical values
df["Description"] = df["Description"].fillna("Unknown")

# Fill missing CustomerID with -1
df["CustomerID"] = df["CustomerID"].fillna(-1)

print(df.isnull().sum())

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


## Find duplicate records

In [18]:
print("Duplicate Records:", df.duplicated().sum())

print(df[df.duplicated()])

df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

Duplicate Records: 5268
       InvoiceNo StockCode                        Description  Quantity  \
517       536409     21866        UNION JACK FLAG LUGGAGE TAG         1   
527       536409     22866      HAND WARMER SCOTTY DOG DESIGN         1   
537       536409     22900    SET 2 TEA TOWELS I LOVE LONDON          1   
539       536409     22111       SCOTTIE DOG HOT WATER BOTTLE         1   
555       536412     22327  ROUND SNACK BOXES SET OF 4 SKULLS         1   
...          ...       ...                                ...       ...   
541675    581538     22068        BLACK PIRATE TREASURE CHEST         1   
541689    581538     23318     BOX OF 6 MINI VINTAGE CRACKERS         1   
541692    581538     22992             REVOLVER WOODEN RULER          1   
541699    581538     22694                       WICKER STAR          1   
541701    581538     23343       JUMBO BAG VINTAGE CHRISTMAS          1   

             InvoiceDate  UnitPrice  CustomerID         Country  
517     0

## Find outliers using IQR

In [19]:
sales_df = df[
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()

print(sales_df.shape)

(524878, 8)


In [20]:
numerical_columns = ["Quantity", "UnitPrice"]

for col in numerical_columns:

    Q1 = sales_df[col].quantile(0.25)
    Q2 = sales_df[col].quantile(0.50)
    Q3 = sales_df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = sales_df[
        (sales_df[col] < lower_limit) |
        (sales_df[col] > upper_limit)
    ]

    print("\nColumn:", col)
    print("Q1:", Q1)
    print("Q2 (Median):", Q2)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower Limit:", lower_limit)
    print("Upper Limit:", upper_limit)
    print("Number of Outliers:", len(outliers))


Column: Quantity
Q1: 1.0
Q2 (Median): 4.0
Q3: 11.0
IQR: 10.0
Lower Limit: -14.0
Upper Limit: 26.0
Number of Outliers: 27111

Column: UnitPrice
Q1: 1.25
Q2 (Median): 2.08
Q3: 4.13
IQR: 2.88
Lower Limit: -3.0700000000000003
Upper Limit: 8.45
Number of Outliers: 37827


## Handle outliers using capping

In [23]:
for col in numerical_columns:

    Q1 = sales_df[col].quantile(0.25)
    Q3 = sales_df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    sales_df[col] = sales_df[col].clip(
        lower=lower_limit,
        upper=upper_limit
    )
